In [1]:
!pip install jdatetime

In [2]:
!pip install pyarrow

In [3]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path
import jdatetime
import pyarrow

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [4]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PRODUCTS_PATH = RAW_DIR / "digikala-products.csv"
COMMENTS_PATH = RAW_DIR / "digikala-comments.csv"

In [5]:
products = pd.read_csv(
    PRODUCTS_PATH,
    low_memory=False
)

comments = pd.read_csv(
    COMMENTS_PATH,
    low_memory=False
)

print("Products:", products.shape)
print("Comments:", comments.shape)

Products: (1283496, 12)
Comments: (6156289, 15)


In [6]:
duplicate_count = products.duplicated().sum()

print("Duplicate rows:", duplicate_count)
print("Duplicate percentage:", duplicate_count / len(products) * 100)

Duplicate rows: 323129
Duplicate percentage: 25.175692016180808


In [7]:
products[
    products.duplicated(keep=False)
].sort_values("id").head(20)

,id,title_fa,Rate,Rate_cnt,Category1,Category2,Brand,Price,Seller,Is_Fake,min_price_last_month,sub_category
1057440,19287,آلبوم موسیقی آتشی در نیستان اثر شهرام ناظری,90,19,موسیقی با کلام,NaN,چهار باغ,1200000,استریو تومبا,False,0,book & stationary & art
1057447,19287,آلبوم موسیقی آتشی در نیستان اثر شهرام ناظری,90,19,موسیقی با کلام,NaN,چهار باغ,1200000,استریو تومبا,False,0,book & stationary & art
1056766,22572,آلبوم موسیقی عشق است اثر ناصر عبداللهی و امیر ...,72,15,موسیقی با کلام,NaN,دارینوش,1000000,استریو تومبا,False,0,book & stationary & art
1056734,22572,آلبوم موسیقی عشق است اثر ناصر عبداللهی و امیر ...,72,15,موسیقی با کلام,NaN,دارینوش,1000000,استریو تومبا,False,0,book & stationary & art
320417,27925,ساعت مچی عقربه ای مردانه فستینا مدل F16600/5,0,0,اکسسوری مردانه,ساعت مردانه,فستینا,131000000,گالری مختاری نوا,False,0,clothe
352360,27925,ساعت مچی عقربه ای مردانه فستینا مدل F16600/5,0,0,اکسسوری مردانه,ساعت مردانه,فستینا,131000000,گالری مختاری نوا,False,0,clothe
351015,31497,ساعت مچی عقربه ای مردانه فستینا F16393/3,0,0,اکسسوری مردانه,ساعت مردانه,فستینا,78000000,گالری مختاری نوا,False,0,clothe
319079,31497,ساعت مچی عقربه ای مردانه فستینا F16393/3,0,0,اکسسوری مردانه,ساعت مردانه,فستینا,78000000,گالری مختاری نوا,False,0,clothe
1198811,37255,کتاب مرکز توجه اثر برایان تریسی,86,3,کتاب فلسفه و روانشناسی,NaN,انتشارات ذهن آویز,750000,خرد و اندیشه,False,649400,book & stationary & art
1198941,37255,کتاب مرکز توجه اثر برایان تریسی,86,3,کتاب فلسفه و روانشناسی,NaN,انتشارات ذهن آویز,750000,خرد و اندیشه,False,649400,book & stationary & art


In [8]:
products = products.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", products.shape)
print("Remaining duplicate rows:", products.duplicated().sum())

Shape after removing duplicates: (960367, 12)
Remaining duplicate rows: 0


In [9]:
duplicate_id_rows = products[
    products["id"].duplicated(keep=False)
].sort_values("id")

print("Rows with duplicated IDs:", len(duplicate_id_rows))
print("Number of duplicated IDs:", duplicate_id_rows["id"].nunique())

duplicate_id_rows.head(30)

Rows with duplicated IDs: 23800
Number of duplicated IDs: 11785


,id,title_fa,Rate,Rate_cnt,Category1,Category2,Brand,Price,Seller,Is_Fake,min_price_last_month,sub_category
233692,12298,ساعت مچی عقربه ای مردانه کاسیو جی شاک GA-110-1ADR,64,21,اکسسوری مردانه,ساعت مردانه,کاسیو,57312000,دیجی‌کالا,False,57133000,clothe
234897,12298,ساعت مچی عقربه ای مردانه کاسیو جی شاک GA-110-1ADR,64,21,اکسسوری مردانه,ساعت مردانه,کاسیو,70832000,دیجی‌کالا,False,57133000,clothe
234876,26147,ساعت مچی عقربه ای مردانه کاسیو کاسیو-GA-100A-9A,80,2,اکسسوری مردانه,ساعت مردانه,کاسیو,62444000,دیجی‌کالا,False,52236000,clothe
233437,26147,ساعت مچی عقربه ای مردانه کاسیو کاسیو-GA-100A-9A,80,2,اکسسوری مردانه,ساعت مردانه,کاسیو,52236000,دیجی‌کالا,False,52236000,clothe
233849,35625,ساعت مچی عقربه ای مردانه کاسیو ادیفایس EFR-526...,92,8,اکسسوری مردانه,ساعت مردانه,کاسیو,58760000,پوزیترون,False,44240000,clothe
234754,35625,ساعت مچی عقربه ای مردانه کاسیو ادیفایس EFR-526...,92,8,اکسسوری مردانه,ساعت مردانه,کاسیو,55566000,پوزیترون,False,44240000,clothe
876442,37279,کتاب فروغ فرخزاد اثر محمد حقوقی نشر نگاه,94,8,کتاب شعر و ادبیات,NaN,نشر نگاه,982100,کتابکالا,False,0,book & stationary & art
876545,37279,کتاب فروغ فرخزاد اثر محمد حقوقی نشر نگاه,94,8,کتاب شعر و ادبیات,NaN,نشر نگاه,982900,کتابکالا,False,0,book & stationary & art
897843,38607,کتاب هوش هیجانی اثر دانیل گولمن نشر نسل نواندیش,86,137,کتاب فلسفه و روانشناسی,NaN,نشر نسل نواندیش,1699000,بلک تام کت,False,0,book & stationary & art
893111,38607,کتاب هوش هیجانی اثر دانیل گولمن نشر نسل نواندیش,86,137,کتاب فلسفه و روانشناسی,NaN,نشر نسل نواندیش,2650000,کتابدان,False,0,book & stationary & art


In [10]:
conflicts = (
    duplicate_id_rows
    .groupby("id")
    .nunique(dropna=False)
    .gt(1)
    .sum()
    .sort_values(ascending=False)
)

conflicts

Price                   8254
Seller                  4036
Rate_cnt                3251
min_price_last_month    1458
sub_category             757
Rate                     646
title_fa                   6
Brand                      2
Category2                  0
Category1                  0
Is_Fake                    0
dtype: int64

In [11]:
null_summary = pd.DataFrame({
    "null_count": products.isnull().sum(),
    "null_percent": products.isnull().mean() * 100
}).sort_values("null_count", ascending=False)

null_summary

,null_count,null_percent
Category2,181890,18.939635
Seller,198,0.020617
id,0,0.000000
title_fa,0,0.000000
Rate_cnt,0,0.000000
Rate,0,0.000000
Brand,0,0.000000
Category1,0,0.000000
Price,0,0.000000
Is_Fake,0,0.000000


In [12]:
products["Category2"] = products["Category2"].fillna("Unknown")
products["Seller"] = products["Seller"].fillna("Unknown")

In [13]:
print(products.isnull().sum())

id                      0
title_fa                0
Rate                    0
Rate_cnt                0
Category1               0
Category2               0
Brand                   0
Price                   0
Seller                  0
Is_Fake                 0
min_price_last_month    0
sub_category            0
dtype: int64


In [14]:
numeric_cols = [
    "Rate",
    "Rate_cnt",
    "Price",
    "min_price_last_month"
]

products[numeric_cols].describe()

,Rate,Rate_cnt,Price,min_price_last_month
count,960367.000000,960367.000000,9.603670e+05,9.603670e+05
mean,30.833221,20.935757,9.133201e+06,4.655652e+05
std,40.141616,218.993884,5.162435e+07,1.123545e+07
min,0.000000,0.000000,0.000000e+00,0.000000e+00
25%,0.000000,0.000000,7.380000e+05,0.000000e+00
50%,0.000000,0.000000,1.700000e+06,0.000000e+00
75%,80.000000,2.000000,4.580000e+06,0.000000e+00
max,100.000000,30438.000000,8.499990e+09,7.645500e+09


In [15]:
for col in numeric_cols:
    print(f"\n{col}")
    print("Negative:", (products[col] < 0).sum())
    print("Zero:", (products[col] == 0).sum())


Rate
Negative: 0
Zero: 590049

Rate_cnt
Negative: 0
Zero: 590034

Price
Negative: 0
Zero: 198

min_price_last_month
Negative: 0
Zero: 900650


In [16]:
print("Rate min:", products["Rate"].min())
print("Rate max:", products["Rate"].max())

Rate min: 0
Rate max: 100


In [17]:
zero_price = products[products["Price"] == 0]

print("Zero price rows:", len(zero_price))

zero_price[
    ["id", "title_fa", "Price", "Seller", "min_price_last_month"]
].head(30)

Zero price rows: 198


,id,title_fa,Price,Seller,min_price_last_month
466,4880138,صابون لیفت ابرو این تاپ مدل S151 به همراه برس ...,0,Unknown,0
22749,1815362,رنگ مو جیپسی شماره 10.0 حجم 100 میلی لیتر رنگ ...,0,Unknown,0
25689,4057774,شامپو رنگ استارلیدی شماره 10.4 حجم 300 میلی لی...,0,Unknown,0
32008,6306435,قلم پاک سازی پوست مدل GOODtime Wisdom,0,Unknown,7990000
44950,8531318,ست اسباب بازی تجهیزات پزشکی دکترتویز کد 01,0,Unknown,0
44951,7602369,هواپیما بازی مدل SUPER WINGS مجموعه 4 عددی,0,Unknown,0
53384,11608311,ساختنی مدل ماینکرافت کد 1076-2,0,Unknown,0
58417,2935826,ماشین بازی جادا مدل NISSAN GTR,0,Unknown,0
59175,813512,عروسک اسب پونی مدل Unicorn01 ارتفاع 38 سانتیمتر,0,Unknown,0
59812,12118507,اسباب بازی مدل فست فود طرح پیتزا کد 02,0,Unknown,0


In [18]:
zero_price_ids = products.loc[
    products["Price"] == 0, "id"
].unique()

has_valid_price = (
    products[
        products["id"].isin(zero_price_ids)
        & (products["Price"] > 0)
    ]["id"]
    .nunique()
)

print("Zero-price product IDs:", len(zero_price_ids))
print("IDs with another valid price:", has_valid_price)
print(
    "IDs with no valid price:",
    len(zero_price_ids) - has_valid_price
)

Zero-price product IDs: 198
IDs with another valid price: 42
IDs with no valid price: 156


In [19]:
products["Price"] = products["Price"].replace(0, np.nan)

In [20]:
print("Missing Price:", products["Price"].isna().sum())
print("Zero Price:", (products["Price"] == 0).sum())

Missing Price: 198
Zero Price: 0


In [21]:
zero_last_month = products["min_price_last_month"] == 0

print("Zero min_price_last_month:", zero_last_month.sum())

print(
    "Zero last month but valid current price:",
    ((products["min_price_last_month"] == 0) & (products["Price"].notna())).sum()
)

print(
    "Both current price and last month price missing/zero:",
    ((products["min_price_last_month"] == 0) & (products["Price"].isna())).sum()
)

Zero min_price_last_month: 900650
Zero last month but valid current price: 900473
Both current price and last month price missing/zero: 177


In [22]:
products["min_price_last_month"] = (
    products["min_price_last_month"]
    .replace(0, np.nan)
)

In [23]:
print(
    "Missing min_price_last_month:",
    products["min_price_last_month"].isna().sum()
)

print(
    "Zero min_price_last_month:",
    (products["min_price_last_month"] == 0).sum()
)

Missing min_price_last_month: 900650
Zero min_price_last_month: 0


In [24]:
text_cols = [
    "title_fa",
    "Category1",
    "Category2",
    "Brand",
    "Seller",
    "sub_category"
]

for col in text_cols:
    s = products[col].astype(str)

    print(f"\n{col}")
    print("Leading/trailing whitespace:", (s != s.str.strip()).sum())
    print("Arabic ي:", s.str.contains("ي", regex=False).sum())
    print("Arabic ك:", s.str.contains("ك", regex=False).sum())
    print("Multiple spaces:", s.str.contains(r"\s{2,}", regex=True).sum())


title_fa
Leading/trailing whitespace: 54480
Arabic ي: 10086
Arabic ك: 4810
Multiple spaces: 33096

Category1
Leading/trailing whitespace: 0
Arabic ي: 0
Arabic ك: 0
Multiple spaces: 0

Category2
Leading/trailing whitespace: 0
Arabic ي: 0
Arabic ك: 0
Multiple spaces: 0

Brand
Leading/trailing whitespace: 97
Arabic ي: 961
Arabic ك: 133
Multiple spaces: 3938

Seller
Leading/trailing whitespace: 24662
Arabic ي: 1606
Arabic ك: 571
Multiple spaces: 2346

sub_category
Leading/trailing whitespace: 0
Arabic ي: 0
Arabic ك: 0
Multiple spaces: 0


In [25]:
for col in ["title_fa", "Brand", "Seller"]:
    products[col] = (
        products[col]
        .str.replace("ي", "ی", regex=False)
        .str.replace("ك", "ک", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

In [26]:
for col in ["title_fa", "Brand", "Seller"]:
    s = products[col].astype(str)

    print(f"\n{col}")
    print("Leading/trailing whitespace:", (s != s.str.strip()).sum())
    print("Arabic ي:", s.str.contains("ي", regex=False).sum())
    print("Arabic ك:", s.str.contains("ك", regex=False).sum())
    print("Multiple spaces:", s.str.contains(r"\s{2,}", regex=True).sum())


title_fa
Leading/trailing whitespace: 0
Arabic ي: 0
Arabic ك: 0
Multiple spaces: 0

Brand
Leading/trailing whitespace: 0
Arabic ي: 0
Arabic ك: 0
Multiple spaces: 0

Seller
Leading/trailing whitespace: 0
Arabic ي: 0
Arabic ك: 0
Multiple spaces: 0


In [27]:
duplicate_count = comments.duplicated().sum()

print("Duplicate rows:", duplicate_count)
print(
    "Duplicate percentage:",
    duplicate_count / len(comments) * 100
)

Duplicate rows: 1318
Duplicate percentage: 0.021409001429270134


In [28]:
duplicate_id_rows = comments[
    comments["id"].duplicated(keep=False)
].sort_values("id")

print("Rows with duplicated IDs:", len(duplicate_id_rows))
print(
    "Number of duplicated IDs:",
    duplicate_id_rows["id"].nunique()
)

Rows with duplicated IDs: 6455
Number of duplicated IDs: 3226


In [29]:
comments = comments.drop_duplicates().reset_index(drop=True)

print("Remaining exact duplicates:", comments.duplicated().sum())

Remaining exact duplicates: 0


In [30]:
duplicate_id_rows = comments[
    comments["id"].duplicated(keep=False)
].sort_values("id")

print("Rows with duplicated IDs:", len(duplicate_id_rows))
print("Number of duplicated IDs:", duplicate_id_rows["id"].nunique())

Rows with duplicated IDs: 3819
Number of duplicated IDs: 1908


In [31]:
conflicts = (
    duplicate_id_rows
    .groupby("id")
    .nunique(dropna=False)
    .gt(1)
    .sum()
    .sort_values(ascending=False)
)

conflicts

likes                    1217
dislikes                  294
body                      241
seller_title              154
seller_code               147
recommendation_status     101
title                      85
rate                       79
advantages                 41
is_buyer                   22
disadvantages               6
true_to_size_rate           4
created_at                  0
product_id                  0
dtype: int64

In [32]:
comments["_engagement"] = (
    comments["likes"].fillna(0)
    + comments["dislikes"].fillna(0)
)

comments = (
    comments
    .sort_values("_engagement", ascending=False)
    .drop_duplicates(subset="id", keep="first")
    .drop(columns="_engagement")
    .reset_index(drop=True)
)

In [33]:
print("Duplicate rows:", comments.duplicated().sum())
print("Duplicate IDs:", comments["id"].duplicated().sum())

Duplicate rows: 0
Duplicate IDs: 0


In [34]:
null_summary = pd.DataFrame({
    "null_count": comments.isnull().sum(),
    "null_percent": comments.isnull().mean() * 100
}).sort_values("null_count", ascending=False)

null_summary[null_summary["null_count"] > 0]

,null_count,null_percent
true_to_size_rate,6062337,98.525563
disadvantages,5741019,93.303478
advantages,5445250,88.496618
title,2863526,46.538243
recommendation_status,894082,14.530689
seller_title,301982,4.907834
seller_code,301982,4.907834
body,637,0.010353


In [35]:
body_null = comments[comments["body"].isna()]

print("Body null rows:", len(body_null))

print(
    "No text anywhere:",
    (
        body_null["title"].isna()
        & body_null["advantages"].isna()
        & body_null["disadvantages"].isna()
    ).sum()
)

print(
    "Has some other text:",
    (
        body_null["title"].notna()
        | body_null["advantages"].notna()
        | body_null["disadvantages"].notna()
    ).sum()
)

Body null rows: 637
No text anywhere: 17
Has some other text: 620


In [36]:
text_cols = [
    "body",
    "title",
    "advantages",
    "disadvantages",
    "seller_title"
]

for col in text_cols:
    s = comments[col].dropna().astype(str)

    print(f"\n{col}")
    print("Leading/trailing whitespace:", (s != s.str.strip()).sum())
    print("Arabic ي:", s.str.contains("ي", regex=False).sum())
    print("Arabic ك:", s.str.contains("ك", regex=False).sum())
    print("Multiple spaces:", s.str.contains(r"\s{2,}", regex=True).sum())


body
Leading/trailing whitespace: 894696
Arabic ي: 48695
Arabic ك: 25348
Multiple spaces: 572394

title
Leading/trailing whitespace: 327886
Arabic ي: 16057
Arabic ك: 7448
Multiple spaces: 18417

advantages
Leading/trailing whitespace: 0
Arabic ي: 9654
Arabic ك: 5518
Multiple spaces: 6197

disadvantages
Leading/trailing whitespace: 0
Arabic ي: 4267
Arabic ك: 2529
Multiple spaces: 3461

seller_title
Leading/trailing whitespace: 169199
Arabic ي: 9619
Arabic ك: 3108
Multiple spaces: 12804


In [37]:
text_cols = [
    "body",
    "title",
    "advantages",
    "disadvantages",
    "seller_title"
]

for col in text_cols:
    comments[col] = (
        comments[col]
        .str.replace("ي", "ی", regex=False)
        .str.replace("ك", "ک", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

In [38]:
for col in text_cols:
    s = comments[col].dropna().astype(str)

    print(f"\n{col}")
    print("Leading/trailing whitespace:", (s != s.str.strip()).sum())
    print("Arabic ي:", s.str.contains("ي", regex=False).sum())
    print("Arabic ك:", s.str.contains("ك", regex=False).sum())
    print("Multiple spaces:", s.str.contains(r"\s{2,}", regex=True).sum())


body
Leading/trailing whitespace: 0
Arabic ي: 0
Arabic ك: 0
Multiple spaces: 0

title
Leading/trailing whitespace: 0
Arabic ي: 0
Arabic ك: 0
Multiple spaces: 0

advantages
Leading/trailing whitespace: 0
Arabic ي: 0
Arabic ك: 0
Multiple spaces: 0

disadvantages
Leading/trailing whitespace: 0
Arabic ي: 0
Arabic ك: 0
Multiple spaces: 0

seller_title
Leading/trailing whitespace: 0
Arabic ي: 0
Arabic ك: 0
Multiple spaces: 0


In [39]:
numeric_cols = [
    "rate",
    "likes",
    "dislikes"
]

comments[numeric_cols].describe()

,rate,likes,dislikes
count,6.153060e+06,6.153060e+06,6.153060e+06
mean,3.648209e+00,4.496592e-01,8.678414e-02
std,1.862066e+00,2.258672e+00,8.305199e-01
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.000000e+00,0.000000e+00,0.000000e+00
50%,4.000000e+00,0.000000e+00,0.000000e+00
75%,5.000000e+00,0.000000e+00,0.000000e+00
max,2.500000e+03,1.136000e+03,3.360000e+02


In [40]:
for col in numeric_cols:
    print(f"\n{col}")
    print("Negative:", (comments[col] < 0).sum())
    print("Zero:", (comments[col] == 0).sum())


rate
Negative: 0
Zero: 529023

likes
Negative: 0
Zero: 4959406

dislikes
Negative: 0
Zero: 5862189


In [41]:
print("Rate min:", comments["rate"].min())
print("Rate max:", comments["rate"].max())

Rate min: 0.0
Rate max: 2500.0


In [42]:
invalid_rate = comments[comments["rate"] > 5]

print("Rate > 5:", len(invalid_rate))
print("\nMost common invalid values:")
print(invalid_rate["rate"].value_counts().head(20))

Rate > 5: 1

Most common invalid values:
rate
2500.0    1
Name: count, dtype: int64


In [43]:
comments.loc[comments["rate"] > 5, "rate"] = np.nan

In [44]:
print(
    comments["recommendation_status"]
    .value_counts(dropna=False)
)

recommendation_status
recommended        4160753
NaN                 894082
no_idea             593715
not_recommended     504510
Name: count, dtype: int64


In [45]:
matched = comments["product_id"].isin(products["id"])

print("Comments with valid product:", matched.sum())
print("Comments without matching product:", (~matched).sum())
print("Match percentage:", matched.mean() * 100)

Comments with valid product: 6153060
Comments without matching product: 0
Match percentage: 100.0


In [46]:
print("Missing created_at:", comments["created_at"].isna().sum())
print("Unique dates:", comments["created_at"].nunique())

comments["created_at"].sample(20, random_state=42).tolist()

Missing created_at: 0
Unique dates: 2654


['22 مرداد 1401',
 '23 تیر 1399',
 '9 دی 1401',
 '8 مهر 1402',
 '11 خرداد 1401',
 '4 اردیبهشت 1402',
 '22 مرداد 1402',
 '22 تیر 1401',
 '23 آبان 1400',
 '5 خرداد 1401',
 '16 مرداد 1401',
 '22 آبان 1396',
 '6 مرداد 1400',
 '18 اردیبهشت 1401',
 '20 مرداد 1402',
 '1 مرداد 1399',
 '13 خرداد 1400',
 '28 اسفند 1400',
 '6 فروردین 1401',
 '28 شهریور 1400']

In [47]:
months = (
    "فروردین|اردیبهشت|خرداد|تیر|مرداد|شهریور|"
    "مهر|آبان|آذر|دی|بهمن|اسفند"
)

pattern = rf"^\d{{1,2}} ({months}) \d{{4}}$"

valid_date_format = comments["created_at"].str.match(pattern)

print("Valid format:", valid_date_format.sum())
print("Invalid format:", (~valid_date_format).sum())

Valid format: 6153060
Invalid format: 0


In [48]:
month_map = {
    "فروردین": 1,
    "اردیبهشت": 2,
    "خرداد": 3,
    "تیر": 4,
    "مرداد": 5,
    "شهریور": 6,
    "مهر": 7,
    "آبان": 8,
    "آذر": 9,
    "دی": 10,
    "بهمن": 11,
    "اسفند": 12,
}

def jalali_to_gregorian(date_str):
    day, month_name, year = date_str.split()

    gregorian = jdatetime.date(
        int(year),
        month_map[month_name],
        int(day)
    ).togregorian()

    return pd.Timestamp(gregorian)

In [49]:
unique_dates = comments["created_at"].unique()

date_mapping = {
    date: jalali_to_gregorian(date)
    for date in unique_dates
}

In [50]:
comments["created_at_gregorian"] = (
    comments["created_at"]
    .map(date_mapping)
)

comments["created_at_gregorian"] = pd.to_datetime(
    comments["created_at_gregorian"]
)

In [51]:
print("Missing converted dates:",
      comments["created_at_gregorian"].isna().sum())

print("Min date:",
      comments["created_at_gregorian"].min())

print("Max date:",
      comments["created_at_gregorian"].max())

comments[
    ["created_at", "created_at_gregorian"]
].sample(10, random_state=42)

Missing converted dates: 0
Min date: 2016-07-13 00:00:00
Max date: 2023-10-18 00:00:00


,created_at,created_at_gregorian
3798809,22 مرداد 1401,2022-08-13
2900550,23 تیر 1399,2020-07-13
1046982,9 دی 1401,2022-12-30
1973931,8 مهر 1402,2023-09-30
5354065,11 خرداد 1401,2022-06-01
5685404,4 اردیبهشت 1402,2023-04-24
2996329,22 مرداد 1402,2023-08-13
3369496,22 تیر 1401,2022-07-13
4719392,23 آبان 1400,2021-11-14
2668190,5 خرداد 1401,2022-05-26


In [53]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

products.to_parquet(
    processed_dir / "products_clean.parquet",
    index=False
)

comments.to_parquet(
    processed_dir / "comments_clean.parquet",
    index=False
)

In [54]:
products_check = pd.read_parquet(
    processed_dir / "products_clean.parquet"
)

comments_check = pd.read_parquet(
    processed_dir / "comments_clean.parquet"
)

print("Products:", products_check.shape)
print("Comments:", comments_check.shape)

Products: (960367, 12)
Comments: (6153060, 16)
